# Phase 7 — Held-Out Patient Validation

**On terminology:** this notebook deliberately says "held-out validation", not "external validation". Phase 1 established that `holdout_validation` (109 patients) and `train_pool` (432 patients + their augmented copies) are a patient-level split of **one** source study population, not two independent cohorts. This still answers a real, non-trivial question — does the frozen model work on patients it never saw, in any form, during training or tuning? — but it is not evidence the model generalizes to a different population, clinic, or measurement process. That distinction is kept explicit throughout.

This is the **first and only** time `holdout_validation` is used. The pipeline below is loaded from disk and only ever `.predict()` / `.predict_proba()`-ed — never refit.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import pandas as pd
import matplotlib.pyplot as plt

from src.config import INTERIM_DIR, TABLES_DIR, MODELS_DIR, PATIENT_ID_COL, TARGET_COL
from src.preprocessing import split_features_target
from src.data_loader import load_clinical_2_full
from src.feature_mapping import harmonize
from src.validate import (
    score_holdout, holdout_metrics_with_ci, build_comparison_table,
    compare_numeric_distributions, compare_categorical_distributions,
)
from src.evaluate import get_confusion_matrix, get_roc_curve, get_pr_curve, get_calibration_curve
from src.viz import apply_chart_style, save_fig, INK_PRIMARY, INK_SECONDARY, GRIDLINE

pd.set_option("display.width", 220)

holdout = pd.read_csv(INTERIM_DIR / "holdout_validation.csv")
X_holdout, y_holdout = split_features_target(holdout)
patient_ids = holdout[PATIENT_ID_COL]

pipeline = joblib.load(MODELS_DIR / "pcos_risk_pipeline.joblib")
print("loaded frozen pipeline:", pipeline.named_steps["model"])
print("holdout shape:", holdout.shape)

## 1. Score the holdout set (once)

In [ ]:
scored = score_holdout(pipeline, X_holdout, y_holdout, patient_ids)
scored.to_csv(TABLES_DIR / "phase7_holdout_scored.csv", index=False)

metrics_ci = holdout_metrics_with_ci(scored, n_boot=1000)
for name, (point, lo, hi) in metrics_ci.items():
    print(f"{name:20s} {point:.3f}  [{lo:.3f}, {hi:.3f}]")

## 2. Comparison table: train_pool CV vs. holdout_validation

In [ ]:
cv_table = pd.read_csv(TABLES_DIR / "phase6_tuned_metrics_with_ci.csv", index_col=0)
cv_row = cv_table.loc["logistic_regression_tuned"]

comparison = build_comparison_table(cv_row, metrics_ci)
comparison.to_csv(TABLES_DIR / "phase7_train_vs_holdout_comparison.csv")
comparison.round(3)

**Interpretation:** every metric on the 109-patient holdout is within a few points of the cross-validated training estimate — recall 0.855→0.861 (virtually identical), ROC-AUC 0.942→0.939, F1 0.799→0.827, PR-AUC 0.905→0.932. Nothing dropped; a couple of metrics (F1, PR-AUC, precision) are a little *higher* on holdout than in CV.

**Why holdout can look slightly better than CV, and why that isn't suspicious:** the frozen pipeline was refit on all 2,029 rows of `train_pool` (Phase 6) after tuning, while each CV fold during tuning only ever trained on ~4/5 of that data — the final model has genuinely seen more data than any individual CV fold's model did. Combined with a small holdout (109 patients, 36 positive) giving noisy point estimates — visible in the wide CIs, e.g. recall's 95% CI is [0.735, 0.969] — a small favorable swing is well within normal sampling variation, not a red flag.

**What this does and doesn't show:** the model performs consistently on patients excluded from every stage of training and tuning. It does not show the model would perform this well on a genuinely different population, clinical site, or measurement protocol — `holdout_validation` cannot answer that question, only a real independent dataset could (see Phase 1).

## 3. Confusion matrix, ROC, PR, and calibration on the holdout set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

cm = get_confusion_matrix(scored)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
ax = axes[0]
ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        color = "white" if cm_norm[i, j] > 0.6 else INK_PRIMARY
        ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.0%})", ha="center", va="center", fontsize=11, color=color)
ax.set_xticks([0, 1]); ax.set_xticklabels(["No PCOS", "PCOS"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["No PCOS", "PCOS"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix — holdout (n=109)", fontsize=10)
for spine in ax.spines.values():
    spine.set_visible(False)

ax = axes[1]
frac_pos, mean_pred = get_calibration_curve(scored, n_bins=5)
ax.plot([0, 1], [0, 1], linestyle="--", color=GRIDLINE, linewidth=1.5, label="Perfectly calibrated")
ax.plot(mean_pred, frac_pos, marker="o", color="#2a78d6", linewidth=1.8, label="Logistic Regression")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed PCOS rate")
ax.set_title("Calibration — holdout, 5 bins (n=109)", fontsize=10)
ax.legend(frameon=False, fontsize=8.5)
apply_chart_style(ax)

save_fig(fig, "18_holdout_confusion_and_calibration")
plt.show()

**Interpretation:** 5 false negatives out of 36 true PCOS patients (86.1% recall) and 8 false positives out of 73 true negatives (89.0% specificity) — consistent with the CV estimate. Calibration with only 5 bins (fewer than Phase 5/6's 10, since 109 patients don't support finer bins reliably) roughly tracks the diagonal, with the same tendency toward mild overconfidence in the mid-range noted in Phase 6 — small-sample noise likely explains the rest.

## 4. Dataset-shift check: do train and holdout patients actually look alike?

Compares the 432 **unaugmented** train patients against the 109 holdout patients (both straight from `clinical_2`, one row per patient — the augmented `lifestyle.csv` copies are excluded here so this is patient-level, not row-level, exactly like Phase 2's EDA). Given both groups are a random stratified split of the *same* population, the expectation is **no meaningful shift** — this section is a sanity check on the split, not a search for a real population difference.

In [ ]:
patient_split = pd.read_csv(INTERIM_DIR / "patient_split.csv")
c2 = harmonize(load_clinical_2_full())
train_ids = set(patient_split.loc[patient_split["split"] == "train", PATIENT_ID_COL])
test_ids = set(patient_split.loc[patient_split["split"] == "test", PATIENT_ID_COL])
train_patients = c2[c2[PATIENT_ID_COL].isin(train_ids)]
holdout_patients = c2[c2[PATIENT_ID_COL].isin(test_ids)]

numeric_cols = [" Age (yrs)", "Weight (Kg)", "BMI", "AMH(ng/mL)", "FSH/LH", "Follicle No. (L)", "Waist:Hip Ratio"]
ks_results = compare_numeric_distributions(train_patients, holdout_patients, numeric_cols)
ks_results.to_csv(TABLES_DIR / "phase7_numeric_shift_ks_test.csv", index=False)
print("Kolmogorov-Smirnov test (numeric features):")
ks_results.round(4)

In [ ]:
cat_cols = ["Cycle(R/I)", "Weight gain(Y/N)", "hair growth(Y/N)", "Skin darkening (Y/N)", TARGET_COL]
chi2_results = compare_categorical_distributions(train_patients, holdout_patients, cat_cols)
chi2_results.to_csv(TABLES_DIR / "phase7_categorical_shift_chi2_test.csv", index=False)
print("Chi-square test (categorical/binary features):")
chi2_results.round(4)

**Interpretation:** every KS test and chi-square test has p > 0.05 — no statistically significant distribution difference found on any checked feature, including the PCOS rate itself (p=1.0, confirming the stratified split preserved class balance exactly). This is the expected, reassuring result for a within-population split: it confirms the split didn't accidentally create two different-looking groups, which would have muddied the metric comparison above. It is **not** evidence of cross-population generalization — both groups are drawn from the same 541-patient study by construction.

## Summary

- The frozen Logistic Regression pipeline was scored exactly once on `holdout_validation`, with no retraining.
- Every metric held up: recall 0.861 [0.735, 0.969], ROC-AUC 0.939 [0.871, 0.988], F1 0.827 [0.727, 0.909] — consistent with (in a couple of cases slightly better than) the cross-validated training estimate.
- No dataset shift was found between the train and holdout patient groups — expected, since both are a stratified split of one population, and this is explicitly **not** claimed as evidence of generalization to an independent cohort.
- The 109-patient holdout is small; confidence intervals are correspondingly wide. Treat the point estimates as indicative, not precise.

**For the report:** state plainly that this is held-out-patient validation within a single source study, not external validation across populations — and that a genuinely independent dataset was not available for this project (Phase 1's finding). This is a limitation to name directly, not a caveat to bury in a footnote.